In [0]:


import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)


class ModelServingDeployer:
    def __init__(self, endpoint_name: str, model_name: str, model_version: str):
        self.endpoint_name = endpoint_name
        self.model_name = model_name
        self.model_version = model_version
        self.client = WorkspaceClient()

    def deploy(self):
        """Endpoint create karta hai agar already exist nahi karta."""
        existing = [e.name for e in self.client.serving_endpoints.list()]
        if self.endpoint_name in existing:
            print(f"Endpoint '{self.endpoint_name}' already exists, skipping creation.")
            return

        self.client.serving_endpoints.create(
            name=self.endpoint_name,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        entity_name=self.model_name,
                        entity_version=self.model_version,
                        workload_size="Small",
                        scale_to_zero_enabled=True,
                    )
                ]
            ),
        )
        print(f" Endpoint '{self.endpoint_name}' creation started.")

    def wait_until_ready(self, timeout_seconds=600, poll_interval=20):
        """Endpoint 'Ready' hone tak wait karta hai (deployment mein kuch minute lagte hain)."""
        elapsed = 0
        while elapsed < timeout_seconds:
            endpoint = self.client.serving_endpoints.get(self.endpoint_name)
            state = endpoint.state.ready
            print(f"Status: {state} (elapsed: {elapsed}s)")
            if str(state) == "EndpointStateReady.READY":
                print(" Endpoint is READY")
                return True
            time.sleep(poll_interval)
            elapsed += poll_interval
        print(" Timeout — endpoint abhi bhi ready nahi hua, Serving UI mein check karo")
        return False


deployer = ModelServingDeployer(
    endpoint_name="fraudguard-endpoint",
    model_name="workspace.default.fraudguard_xgboost",
    model_version="1",
)
deployer.deploy()

In [0]:
deployer.wait_until_ready()

In [0]:
# Sample transaction manually banate hain (feature columns ke hisab se)
sample_txn = {
    "amount": 9839.64,
    "oldbalanceOrg": 170136.0,
    "newbalanceOrig": 160296.36,
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0,
    "orig_balance_ratio": 9839.64 / 170136.0,
    "orig_drained_to_zero": 0,
    "dest_balance_ratio": 0.0,
    "orig_txn_count_so_far": 1,
    "is_cash_out": 0,
    "is_transfer": 0,
    "is_payment": 1,
    "txn_count_1min": 1,
    "txn_count_5min": 1,
    "txn_count_15min": 1,
    "txn_amount_sum_1min": 9839.64,
    "txn_amount_sum_5min": 9839.64,
    "seconds_since_last_txn": 999999,
    "distinct_merchants_24h": 1,
}
print("Input:", sample_txn)

In [0]:
class FraudPredictor:
    def __init__(self, endpoint_name: str, client):
        self.endpoint_name = endpoint_name
        self.client = client

    def predict(self, transaction_features: dict) -> dict:
        response = self.client.serving_endpoints.query(
            name=self.endpoint_name,
            dataframe_records=[transaction_features],
        )
        return response.predictions


predictor = FraudPredictor("fraudguard-endpoint", deployer.client)
result = predictor.predict(sample_txn)
print("Prediction:", result)

In [0]:
# Fraud-pattern jaisi transaction — bada amount, poora balance drain
fraud_like_txn = {
    "amount": 181.0,
    "oldbalanceOrg": 181.0,
    "newbalanceOrig": 0.0,          # poora balance drain ho gaya
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0,
    "orig_balance_ratio": 1.0,       # amount = poora balance
    "orig_drained_to_zero": 1,       # key fraud signal
    "dest_balance_ratio": 0.0,
    "orig_txn_count_so_far": 1,
    "is_cash_out": 1,
    "is_transfer": 0,
    "is_payment": 0,
    "txn_count_1min": 1,
    "txn_count_5min": 1,
    "txn_count_15min": 1,
    "txn_amount_sum_1min": 181.0,
    "txn_amount_sum_5min": 181.0,
    "seconds_since_last_txn": 5,      # bohat jaldi dobara transaction
    "distinct_merchants_24h": 3,
}

result2 = predictor.predict(fraud_like_txn)
print("Fraud-like prediction:", result2)